In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

device = torch.device('cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
features_path = ('models/evaluation/data/features_normalized.nc')
targets_path = ('models/evaluation/data/incois_argo_thetao_2022_2024_eval.nc')
mask_path = ('data_pipeline/data/processed/ml_ready/ocean_mask_3d.nc')
stats_path = ('data_pipeline/data/processed/ml_ready/normalization_stats.json')

# 3. Lazy Load NetCDF Files using xarray
print("\n=== Dataset Mapping ===")
features_ds = xr.open_dataset(features_path)
targets_ds = xr.open_dataset(targets_path)
mask_ds = xr.open_dataset(mask_path)

with open(stats_path, 'r') as f:
    norm_stats = json.load(f)

print(f"Features mapped: {list(features_ds.data_vars)}")
print(f"Targets mapped: {list(targets_ds.data_vars)}")
print("Normalization stats loaded successfully.")


=== Dataset Mapping ===
Features mapped: ['analysed_sst', 'sos', 'sla', 'uwnd', 'vwnd', 'u', 'v']
Targets mapped: ['thetao', 'argo_obs_mask']
Normalization stats loaded successfully.


In [3]:
xr.set_options(file_cache_maxsize=1)

class StratifiedOceanDataset(Dataset):
    def __init__(self, features_ds, targets_ds, mask_ds, target_years, seq_len=10):
        self.features_ds = features_ds
        self.targets_ds = targets_ds
        self.seq_len = seq_len
        self.feature_vars = ['analysed_sst', 'sos', 'sla', 'uwnd', 'vwnd', 'u', 'v']
        
        # Extract 2D surface mask using 'depth=0': shape [1, H, W]
        surf_mask_2d = mask_ds['ocean_mask'].isel(depth=0).values.astype(np.float32)
        # np.nan_to_num(surf_mask_2d, copy=False, nan=0.0)
        self.mask_target = surf_mask_2d[np.newaxis, :, :] # [1, H, W]
        self.mask_hist = np.tile(surf_mask_2d[np.newaxis, np.newaxis, :, :], (seq_len, 1, 1, 1)) # [T, 1, H, W]
        
        times = pd.to_datetime(features_ds.time.values)
        total_time_len = len(times)
        
        # Filter sequences that fall strictly inside target test years
        self.valid_indices = []
        for idx in range(total_time_len - self.seq_len):
            window_slice_years = times[idx : idx + self.seq_len + 1].year
            if all(y in target_years for y in window_slice_years):
                self.valid_indices.append(idx)
                
    def __len__(self):
        return len(self.valid_indices)
        
    def __getitem__(self, index):
        idx = self.valid_indices[index]
        
        # 1. 7 dynamic vars + 1 surface mask -> 8 channels for 10-Day History [T, 8, H, W]
        hist_slice = self.features_ds.isel(time=slice(idx, idx + self.seq_len))
        x_hist_7 = np.stack([hist_slice[var].values for var in self.feature_vars], axis=1).astype(np.float32)
        np.nan_to_num(x_hist_7, copy=False, nan=0.0)
        x_hist_np = np.concatenate([x_hist_7, self.mask_hist], axis=1)
        
        # 2. 7 dynamic vars + 1 surface mask -> 8 channels for Target Day [8, H, W]
        target_day_slice = self.features_ds.isel(time=idx + self.seq_len)
        x_target_7 = np.stack([target_day_slice[var].values for var in self.feature_vars], axis=0).astype(np.float32)
        np.nan_to_num(x_target_7, copy=False, nan=0.0)
        x_target_np = np.concatenate([x_target_7, self.mask_target], axis=0)
        
        # 3. Target Day 3D Subsurface Temperatures [15, H, W]
        y_target_np = self.targets_ds['thetao'].isel(time=idx + self.seq_len).values.astype(np.float32)
        np.nan_to_num(y_target_np, copy=False, nan=0.0)
        
        del hist_slice, target_day_slice, x_hist_7, x_target_7
        
        return (
            torch.from_numpy(x_hist_np), 
            torch.from_numpy(x_target_np), 
            torch.from_numpy(y_target_np)
        )

# Evaluation on holdout test years
test_years = [2022, 2023, 2024]
test_dataset = StratifiedOceanDataset(features_ds, targets_ds, mask_ds, target_years=test_years, seq_len=10)

# drop_last=False ensures 100% of test days are evaluated; num_workers=0 runs cleanly on local CPU
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=False, drop_last=False)

print(f"Test DataLoader ready! Total evaluation sequences: {len(test_dataset)}")

Test DataLoader ready! Total evaluation sequences: 1070


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Corrected Module 1: Temporal Scale Mixer ---
class TemporalScaleMixer(nn.Module):
    def __init__(self, in_channels=7, branch_channels=16):
        super().__init__()
        self.branch_a = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, kernel_size=(2, 1, 1), stride=(2, 1, 1)),
            nn.BatchNorm3d(branch_channels),
            nn.ReLU(inplace=True)
        )
        self.branch_b = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, kernel_size=(5, 1, 1), stride=(5, 1, 1)),
            nn.BatchNorm3d(branch_channels),
            nn.ReLU(inplace=True)
        )
        self.branch_c = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, kernel_size=(10, 1, 1), stride=(10, 1, 1)),
            nn.BatchNorm3d(branch_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # 1. Swap dimensions from [B, Time, C, H, W] to [B, C, Time, H, W]
        x = x.permute(0, 2, 1, 3, 4)
        
        # 2. Run the branches
        out_a = self.branch_a(x) 
        out_b = self.branch_b(x) 
        out_c = self.branch_c(x) 
        
        # 3. Align time dimensions back to 10 days using interpolation
        target_size = (10, x.shape[3], x.shape[4])
        out_a = F.interpolate(out_a, size=target_size, mode='nearest')
        out_b = F.interpolate(out_b, size=target_size, mode='nearest')
        out_c = F.interpolate(out_c, size=target_size, mode='nearest')
        
        # 4. Concatenate original input with all branch outputs
        mixed_features = torch.cat([x, out_a, out_b, out_c], dim=1)
        
        # 5. Swap dimensions back to [B, Time, C, H, W] for the ConvLSTM
        mixed_features = mixed_features.permute(0, 2, 1, 3, 4)
        
        return mixed_features

print("Module 1 (TemporalScaleMixer) loaded.")

Module 1 (TemporalScaleMixer) loaded.


In [5]:
# --- Module 2: Spatiotemporal ConvLSTM ---

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        # Padding ensures the spatial dimensions (Lat/Lon) stay exactly the same[cite: 7]
        padding = kernel_size // 2
        
        # A single convolution calculates all 4 LSTM gates at once for efficiency[cite: 7]
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim, 
            out_channels=4 * hidden_dim, 
            kernel_size=kernel_size, 
            padding=padding
        )

    def forward(self, x, state):
        h, c = state
        
        # Combine the current day's input with yesterday's hidden state[cite: 7]
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        
        # Split the convolution output into the 4 control gates[cite: 7]
        i, f, g, o = torch.chunk(gates, 4, dim=1)
        
        # Apply standard LSTM activations[cite: 7]
        i = torch.sigmoid(i)       # Input gate
        f = torch.sigmoid(f)       # Forget gate
        g = torch.tanh(g)          # Cell update
        o = torch.sigmoid(o)       # Output gate
        
        # Update cell state (long-term memory) and hidden state (short-term memory)[cite: 7]
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        
        return h_next, c_next

class SpatiotemporalConvLSTM(nn.Module):
    # input_dim must match the output channels of Module 1 (e.g., 7 base vars + 3 branches * 16 = 55)
    def __init__(self, input_dim=55, hidden_dim=32, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = ConvLSTMCell(input_dim, hidden_dim, kernel_size)

    def forward(self, x):
        # x shape expects: (Batch, Time, Channels, Height, Width)[cite: 7]
        B, T, C, H, W = x.shape
        
        # Initialize the first day's memory as empty (zeros)[cite: 7]
        h = torch.zeros(B, self.hidden_dim, H, W, device=x.device)
        c = torch.zeros(B, self.hidden_dim, H, W, device=x.device)
        
        # Loop sequentially through the 10 days[cite: 7]
        for t in range(T):
            h, c = self.cell(x[:, t, :, :, :], (h, c))
            
        # For predicting day 11, we only need the final summarized state at day 10[cite: 7]
        return h

print("Module 2 (SpatiotemporalConvLSTM) loaded.")

Module 2 (SpatiotemporalConvLSTM) loaded.


In [6]:
# Module 3
# --- U-Net Building Blocks ---

class DoubleConv(nn.Module):
    """(Convolution -> BatchNorm -> ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        # Dynamically resize x1 to exactly match the spatial dimensions (Height, Width) of x2[cite: 7]
        x1 = F.interpolate(x1, size=x2.shape[2:], mode='bilinear', align_corners=True)
        
        # Concatenate skip connection (x2) with the perfectly sized upsampled features (x1)[cite: 7]
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

# --- The Bottleneck: HASPP ---

class HASPP(nn.Module):
    """Hierarchical Atrous Spatial Pyramid Pooling"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        dims = out_channels // 5 # Split out_channels into 5 parallel paths[cite: 7]
        
        # Path 1: 1x1 Convolution (Strictly local)[cite: 7]
        self.aspp1 = nn.Sequential(
            nn.Conv2d(in_channels, dims, kernel_size=1, bias=False),
            nn.BatchNorm2d(dims),
            nn.ReLU(inplace=True)
        )
        # Path 2: 3x3 Conv, Dilation 6 (padding=dilation keeps shape same)[cite: 7]
        self.aspp2 = nn.Sequential(
            nn.Conv2d(in_channels, dims, kernel_size=3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(dims),
            nn.ReLU(inplace=True)
        )
        # Path 3: 3x3 Conv, Dilation 12[cite: 7]
        self.aspp3 = nn.Sequential(
            nn.Conv2d(in_channels, dims, kernel_size=3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(dims),
            nn.ReLU(inplace=True)
        )
        # Path 4: 3x3 Conv, Dilation 18[cite: 7]
        self.aspp4 = nn.Sequential(
            nn.Conv2d(in_channels, dims, kernel_size=3, padding=18, dilation=18, bias=False),
            nn.BatchNorm2d(dims),
            nn.ReLU(inplace=True)
        )
        # Path 5: Global Average Pooling (Basin Scale)[cite: 7]
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, dims, kernel_size=1, bias=False),
            nn.BatchNorm2d(dims),
            nn.ReLU(inplace=True)
        )
        
        # Final 1x1 projection to lock the output channels perfectly[cite: 7]
       # Inside HASPP.__init__:
        self.project = nn.Sequential(
            nn.Conv2d(dims * 5, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.15)  # <-- Prevents memorizing training-year spatial noise
        )

    def forward(self, x):
        x1 = self.aspp1(x)
        x2 = self.aspp2(x)
        x3 = self.aspp3(x)
        x4 = self.aspp4(x)
        x5 = self.global_pool(x)
        
        # Upsample global pool path back to original spatial size[cite: 7]
        x5 = F.interpolate(x5, size=x.shape[2:], mode='bilinear', align_corners=True)
        
        # Concatenate all 5 paths and project[cite: 7]
        out = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.project(out)

In [7]:
# --- Final Module 3 (U-Net) ---

class SFFM_UNet(nn.Module):
    def __init__(self, hist_channels=32, target_channels=7, out_depths=15):
        super().__init__()
        # HYPERPARAMETER TO TUNE: Increase 'base' to 64 or 128 for Kaggle T4 GPUs
        base = 64 
        
        # 1. Dual-Branch Spatial Encoder[cite: 7]
        self.inc_h = DoubleConv(hist_channels, base)
        self.inc_x = DoubleConv(target_channels, base)
        
        # 2. Encoder Downsampling Paths[cite: 7]
        self.down1 = Down(base, base * 2)      
        self.down2 = Down(base * 2, base * 4)  
        
        # 3. Bottleneck (HASPP)[cite: 7]
        self.haspp = HASPP(base * 4, base * 4) 
        
        # 4. Decoder Upsampling Paths with Skip Connections[cite: 7]
        self.up1 = Up(base * 4 + base * 2, base * 2) 
        self.up2 = Up(base * 2 + base, base)         
        
        # 5. Output Head: Final Refinement & Projection to Depth[cite: 7]
        self.final_refinement = nn.Sequential(
            DoubleConv(base, base), 
            nn.Conv2d(base, out_depths, kernel_size=1) 
        )

    def forward(self, h_hist, x_target):
        feat_h = self.inc_h(h_hist)
        feat_x = self.inc_x(x_target)
        
        # First Fusion & Skip Connection 1[cite: 7]
        x1 = feat_h + feat_x 
        
        # Encoder Level 2[cite: 7]
        x2 = self.down1(x1)
        
        # Encoder Level 3 & Skip Connection 2[cite: 7]
        x3 = self.down2(x2)
        
        # Bottleneck (HASPP)[cite: 7]
        x3 = self.haspp(x3)
        
        # Decoder Level 2 (Fuse with Skip Connection 2)[cite: 7]
        x = self.up1(x3, x2)
        
        # Decoder Level 1 (Fuse with Skip Connection 1)[cite: 7]
        x = self.up2(x, x1)
        
        # Final non-linear refinement and projection to vertical depths[cite: 7]
        pred_3d = self.final_refinement(x)
        return pred_3d

print("Module 3 (SFFM_UNet) loaded.")

Module 3 (SFFM_UNet) loaded.


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# --- Module 4: Sparse In-Situ ARGO & Physics-Informed Evaluation Engine ---
class ArgoOceanPhysicsLoss(nn.Module):
    def __init__(self, thetao_mean, thetao_std, depth_indices=None, alpha_phys=0.05, beta_grad=0.1):
        super().__init__()
        self.alpha_phys = alpha_phys
        self.beta_grad = beta_grad
        
        # Register normalization stats as buffers on CPU/device to convert pred -> real °C
        self.register_buffer('thetao_mean', torch.tensor(thetao_mean, dtype=torch.float32))
        self.register_buffer('thetao_std', torch.tensor(thetao_std, dtype=torch.float32))
        self.depth_indices = depth_indices # None if 15 depths, or list of 7 indices if 7 depths

    def forward(self, pred_norm, target_argo_c, mask_3d):
        """
        pred_norm:     [B, 15, H, W] (Z-score normalized output from OceanEmbed)
        target_argo_c: [B, D_argo, H, W] (Real °C from INCOIS ARGO, with NaNs at unobserved points)
        mask_3d:       [15, H, W] (Static 3D basin ocean mask)
        """
        batch_size = pred_norm.shape[0]

        # 1. Un-normalize model prediction from Z-score to real Degrees Celsius (°C)
        pred_c = (pred_norm * self.thetao_std) + self.thetao_mean

        # 2. Full-Basin Physics Inversion Check (evaluated across all 15 depths of the ocean)
        mask_batch_full = mask_3d.expand(batch_size, -1, -1, -1)
        diff_z_pred_full = pred_c[:, 1:, :, :] - pred_c[:, :-1, :, :]
        pair_mask_full = mask_batch_full[:, 1:, :, :] * mask_batch_full[:, :-1, :, :]
        valid_basin_pairs = torch.sum(pair_mask_full) + 1e-7
        
        inversion_penalty = F.relu(diff_z_pred_full) ** 2
        loss_phys = torch.sum(inversion_penalty * pair_mask_full) / valid_basin_pairs

        # 3. Align Depth Levels if ARGO target has 7 depths instead of 15
        if self.depth_indices is not None:
            pred_c_aligned = pred_c[:, self.depth_indices, :, :]
            mask_aligned = mask_batch_full[:, self.depth_indices, :, :]
        else:
            pred_c_aligned = pred_c
            mask_aligned = mask_batch_full

        # 4. Build Dynamic Sparse ARGO Observation Mask (1 ONLY where real float data exists)
        argo_obs_mask = (~torch.isnan(target_argo_c)).float() * mask_aligned
        valid_argo_points = torch.sum(argo_obs_mask)

        # Safe-replace NaNs with 0.0 AFTER building argo_obs_mask so NaNs don't poison math
        target_clean_c = torch.nan_to_num(target_argo_c, nan=0.0)

        # 5. Sparse In-Situ Data Loss (MSE in °C² & MAE in °C at real ARGO locations)
        if valid_argo_points > 0:
            diff_c = (pred_c_aligned - target_clean_c) * argo_obs_mask
            loss_data_mse = torch.sum(diff_c ** 2) / valid_argo_points
            loss_data_mae = torch.sum(torch.abs(diff_c)) / valid_argo_points
        else:
            loss_data_mse = torch.tensor(0.0, device=pred_norm.device)
            loss_data_mae = torch.tensor(0.0, device=pred_norm.device)

        # 6. Sparse Vertical Gradient Loss (where two consecutive ARGO depths BOTH have data)
        argo_pair_mask = argo_obs_mask[:, 1:, :, :] * argo_obs_mask[:, :-1, :, :]
        valid_argo_pairs = torch.sum(argo_pair_mask)

        if valid_argo_pairs > 0:
            diff_z_pred_argo = pred_c_aligned[:, 1:, :, :] - pred_c_aligned[:, :-1, :, :]
            diff_z_true_argo = target_clean_c[:, 1:, :, :] - target_clean_c[:, :-1, :, :]
            grad_err = ((diff_z_pred_argo - diff_z_true_argo) ** 2) * argo_pair_mask
            loss_grad = torch.sum(grad_err) / valid_argo_pairs
        else:
            loss_grad = torch.tensor(0.0, device=pred_norm.device)

        total_loss = loss_data_mse + (self.alpha_phys * loss_phys) + (self.beta_grad * loss_grad)
        
        # Return total, MSE (°C²), Physics, Grad, MAE (°C), and count of real ARGO points in batch
        return total_loss, loss_data_mse, loss_phys, loss_grad, loss_data_mae, valid_argo_points.item()


# --- Master Architecture Wrapper (Unchanged so .pth files load cleanly) ---
class OceanEmbed(nn.Module):
    def __init__(self, in_channels=7, mixer_branches=16, lstm_hidden=32, target_channels=7, out_depths=15):
        super().__init__()
        self.ts_mixer = TemporalScaleMixer(
            in_channels=in_channels, 
            branch_channels=mixer_branches
        )
        self.conv_lstm = SpatiotemporalConvLSTM(
            input_dim=in_channels + (3 * mixer_branches), 
            hidden_dim=lstm_hidden
        )
        self.sffm = SFFM_UNet(
            hist_channels=lstm_hidden, 
            target_channels=target_channels, 
            out_depths=out_depths
        )

    def forward(self, x_history, x_target_day):
        mixed_history = self.ts_mixer(x_history)
        summary_map = self.conv_lstm(mixed_history)
        pred_3d = self.sffm(summary_map, x_target_day)
        return pred_3d

print("Module 4 (ArgoOceanPhysicsLoss) and Master Class (OceanEmbed) loaded.")

Module 4 (ArgoOceanPhysicsLoss) and Master Class (OceanEmbed) loaded.


In [9]:
import numpy as np
import pandas as pd

print("=== 1. NORMALIZATION STATS JSON ===")
print(json.dumps(norm_stats, indent=2))

print("\n=== 2. SURFACE FEATURES DATASET (features_ds) ===")
print("Dimensions:", dict(features_ds.dims))
print("Variables :", list(features_ds.data_vars))
feat_times = pd.to_datetime(features_ds.time.values)
print(f"Time Range: {feat_times.min().date()} to {feat_times.max().date()} ({len(feat_times)} days)")
print("Years present:", sorted(list(set(feat_times.year))))

print("\n=== 3. TARGET / ARGO DATASET (targets_ds) ===")
print("Dimensions:", dict(targets_ds.dims))
print("Variables :", list(targets_ds.data_vars))
print("Coords    :", list(targets_ds.coords))
targ_times = pd.to_datetime(targets_ds.time.values)
print(f"Time Range: {targ_times.min().date()} to {targ_times.max().date()} ({len(targ_times)} days)")
print("Years present:", sorted(list(set(targ_times.year))))

# Check depth coordinate name and values in targets_ds
depth_coord_t = 'depth' if 'depth' in targets_ds.coords else ('lev' if 'lev' in targets_ds.coords else None)
if depth_coord_t:
    print(f"Target Depth Coord ('{depth_coord_t}'):", targets_ds[depth_coord_t].values)

# Sample check on target variable values (first 30 days) to verify °C vs Normalized and NaN sparsity
targ_var = list(targets_ds.data_vars)[0]
sample_vals = targets_ds[targ_var].isel(time=slice(0, 30)).values
valid_count = np.sum(~np.isnan(sample_vals))
print(f"Target Var '{targ_var}' (First 30 days) -> Valid Points: {valid_count} / {sample_vals.size} "
      f"| Min: {np.nanmin(sample_vals):.3f}, Max: {np.nanmax(sample_vals):.3f}")

print("\n=== 4. 3D MASK DATASET (mask_ds) ===")
print("Dimensions:", dict(mask_ds.dims))
print("Variables :", list(mask_ds.data_vars))
depth_coord_m = 'depth' if 'depth' in mask_ds.coords else ('lev' if 'lev' in mask_ds.coords else None)
if depth_coord_m:
    print(f"Mask Depth Coord ('{depth_coord_m}'):", mask_ds[depth_coord_m].values)

=== 1. NORMALIZATION STATS JSON ===
{
  "analysed_sst": {
    "mean": 28.171791076660156,
    "std": 1.7956781387329102
  },
  "sos": {
    "mean": 34.701255798339844,
    "std": 1.9610856771469116
  },
  "sla": {
    "mean": 0.058311380445957184,
    "std": 0.09984976798295975
  },
  "uwnd": {
    "mean": 1.2184956073760986,
    "std": 4.682363033294678
  },
  "vwnd": {
    "mean": 0.29616203904151917,
    "std": 4.391159534454346
  },
  "u": {
    "mean": 0.013475114479660988,
    "std": 0.22300511598587036
  },
  "v": {
    "mean": 0.011245391331613064,
    "std": 0.2042849063873291
  },
  "thetao": {
    "mean": 21.2752742767334,
    "std": 7.5230841636657715
  }
}

=== 2. SURFACE FEATURES DATASET (features_ds) ===
Dimensions: {'time': 1080, 'latitude': 100, 'longitude': 240}
Variables : ['analysed_sst', 'sos', 'sla', 'uwnd', 'vwnd', 'u', 'v']
Time Range: 2022-01-01 to 2024-12-15 (1080 days)
Years present: [2022, 2023, 2024]

=== 3. TARGET / ARGO DATASET (targets_ds) ===
Dimensions

/tmp/ipykernel_44957/4212749219.py:8: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Dimensions:", dict(features_ds.dims))
/tmp/ipykernel_44957/4212749219.py:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Dimensions:", dict(targets_ds.dims))


Target Var 'thetao' (First 30 days) -> Valid Points: 3811 / 10800000 | Min: 6.228, Max: 29.677

=== 4. 3D MASK DATASET (mask_ds) ===
Dimensions: {'depth': 15, 'latitude': 100, 'longitude': 240}
Variables : ['ocean_mask']
Mask Depth Coord ('depth'): [   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]


/tmp/ipykernel_44957/4212749219.py:35: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Dimensions:", dict(mask_ds.dims))


In [10]:
# Load the lightweight mask/thetao summary across all 1096 days
argo_mask_all = targets_ds['argo_obs_mask'].values
thetao_all = targets_ds['thetao'].values
targ_years = pd.to_datetime(targets_ds.time.values).year

print(f"Total Valid ARGO Mask Points (2022-2024): {np.nansum(argo_mask_all):.0f}")
print(f"Total Non-NaN thetao Points  (2022-2024): {np.sum(~np.isnan(thetao_all))}")

if np.sum(~np.isnan(thetao_all)) > 0:
    print(f"ARGO thetao Range: Min = {np.nanmin(thetao_all):.2f}°C | Max = {np.nanmax(thetao_all):.2f}°C | Mean = {np.nanmean(thetao_all):.2f}°C")
    for yr in [2022, 2023, 2024]:
        yr_pts = np.sum(~np.isnan(thetao_all[targ_years == yr]))
        yr_days = np.sum(np.any(~np.isnan(thetao_all[targ_years == yr]), axis=(1, 2, 3)))
        print(f"  -> Year {yr}: {yr_pts:,} valid float measurements across {yr_days} days")

Total Valid ARGO Mask Points (2022-2024): 124238
Total Non-NaN thetao Points  (2022-2024): 124238
ARGO thetao Range: Min = 6.00°C | Max = 32.77°C | Mean = 21.02°C
  -> Year 2022: 42,636 valid float measurements across 362 days
  -> Year 2023: 36,134 valid float measurements across 337 days
  -> Year 2024: 45,468 valid float measurements across 366 days


In [16]:
# 1. Install the Python wrappers
!pip install -q torchview graphviz

import torch
import torch.onnx
from torchview import draw_graph
import graphviz

# Jupyter formatting for graphviz
graphviz.set_jupyter_format('png')

# 2. Initialize a blank CPU model
print("Initializing blank OceanEmbed model for tracing...")
model = OceanEmbed(in_channels=8, target_channels=8).to('cpu')
model.eval()

# 3. Create tiny "dummy" tensors to trace the pathways
dummy_hist = torch.randn(1, 10, 8, 100, 240)
dummy_tgt = torch.randn(1, 8, 100, 240)

# ==========================================
# EXPORT 1: Torchview 2D Flowchart (PNG/SVG)
# ==========================================
print("Generating Torchview 2D Flowchart...")
model_graph = draw_graph(
    model, 
    input_data=(dummy_hist, dummy_tgt),
    expand_nested=True, # Expands Module 1, 2, 3 into distinct boxed sections
    depth=3,            # Traces down to the individual convolutions
    save_graph=True,
    filename="oceanembed_architecture_flowchart" 
)
print("Saved 'oceanembed_architecture_flowchart.png' to your folder.")

# ==========================================
# EXPORT 2: ONNX Graph for Vector Editing
# ==========================================
print("Exporting ONNX graph for UI vector editing...")
torch.onnx.export(
    model, 
    (dummy_hist, dummy_tgt), 
    "oceanembed_netron.onnx",
    input_names=["10-Day History (8 Ch)", "Target Day Surface (8 Ch)"],
    output_names=["3D Subsurface Temperatures (15 Depths)"]
)
print("Saved 'oceanembed_netron.onnx' to your folder.")

Initializing blank OceanEmbed model for tracing...
Generating Torchview 2D Flowchart...
Saved 'oceanembed_architecture_flowchart.png' to your folder.
Exporting ONNX graph for UI vector editing...
[torch.onnx] Obtain model graph for `OceanEmbed([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `OceanEmbed([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/home/shrey/anaconda3/envs/Ocean/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved 'oceanembed_netron.onnx' to your folder.


In [18]:
!pip install torchview graphviz

import torch
from torchview import draw_graph

# 1. Initialize your model
model = OceanEmbed(in_channels=8, target_channels=8)

# 2. Create dummy inputs matching your batch size and tensor shapes
# History: [Batch, Time, Channels, Height, Width]
dummy_hist = torch.randn(1, 10, 8, 100, 240)
# Target Day: [Batch, Channels, Height, Width]
dummy_tgt = torch.randn(1, 8, 100, 240)

# 3. Generate the architecture graph
model_graph = draw_graph(
    model, 
    input_data=(dummy_hist, dummy_tgt),
    expand_nested=True, # Expands Module 1, 2, 3 into distinct boxed sections
    depth=3,            # Controls how deep into the sub-layers it draws
    save_graph=True,
    filename="oceanembed_flowchart" 
)

In [20]:
!pip install visualtorch

import visualtorch
import torch

model = OceanEmbed(in_channels=8, target_channels=8)

# Renders the 3D block flow
img = visualtorch.render(
    model, 
    input_shape=[(1, 10, 8, 100, 240), (1, 8, 100, 240)], 
    style="flow",       # 'flow' creates the 3D stacked boxes
    show_neurons=False, 
    layer_spacing=60
)
img.save("oceanembed_3d_architecture.png")

TypeError: FlowStyleOptions.__init__() got an unexpected keyword argument 'show_neurons'